# RigidHitch — embed the catalog images

Runs the **slow step** of the RigidHitch pipeline on a free Colab GPU: turning each
de-duplicated catalog photo into a fingerprint vector. Everything before and after this
runs locally in seconds.

Nothing is trained here. DINOv2 is used frozen and off-the-shelf — each image passes
through once. Expect **45–90 minutes** for the full set.

### Before you start

Run these two locally, then upload the results to Drive under `MyDrive/rigidhitch/`:

```
python scripts/rigidhitch_dedup_images.py       # -> embed_rows.jsonl
python scripts/rigidhitch_pack_for_colab.py     # -> rigidhitch_images_518.zip (~250 MB)
```

### Runtime → Change runtime type → **T4 GPU**

The notebook only *calls* the version-controlled scripts, so the embedding logic stays in
git rather than trapped in a notebook. Do not paste model code in here.

## 1 — Confirm a GPU is actually attached

If this fails, fix the runtime type before going further. On CPU the run takes hours.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run."
)
print(torch.cuda.get_device_name(0))

## 2 — Install what Colab is missing

Torch is preinstalled; only the pinned `transformers` (which loads DINOv2) is needed.

In [ ]:
!pip install -q transformers==4.46.3 pydantic-settings==2.7.1 python-dotenv==1.0.1

## 3 — Mount Drive

Drive holds the input zip and receives the checkpoints, so a disconnect never loses the run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/rigidhitch'
!ls -lh {DRIVE}

## 4 — Unzip to local disk, **not** Drive

This matters more than it looks. Reading 17k small files over Drive's network filesystem
costs more time than the embedding itself. `/content` is local SSD.

In [ ]:
!mkdir -p /content/images
!unzip -q -o {DRIVE}/rigidhitch_images_518.zip -d /content/images
!find /content/images -name '*.jpg' | wc -l

## 5 — Clone the repo at a pinned commit

So the *real* `EmbeddingGenerator` runs — same preprocessing and same TTA setting the
live app applies to a query photo. If these two ever diverge, the vectors stop being
comparable and accuracy degrades silently, with no error to notice.

Set `COMMIT` to the exact revision you want, so a rebuild months from now is reproducible.

In [ ]:
import os
from getpass import getpass

COMMIT = 'dev'  # replace with a commit SHA for a reproducible rebuild
TOKEN = getpass('GitHub token (repo scope): ')

!rm -rf /content/partpilot-repo
!git clone -q https://{TOKEN}@github.com/klizerteam/partpilot.git /content/partpilot-repo
%cd /content/partpilot-repo
!git checkout -q {COMMIT}
%cd /content/partpilot-repo/partpilot

# Copy the row manifest in from Drive — it is the contract between this run and the build.
!mkdir -p /content/build
!cp {DRIVE}/embed_rows.jsonl /content/build/
print(os.getcwd())

## 6 — Smoke test: 200 images

**Do not skip this.** Two minutes here catches a wrong path, a bad setting, or a broken
checkout before you spend an hour discovering it. Check that `dim` is 768 for
`dinov2-base` and that no images failed.

In [ ]:
!python scripts/rigidhitch_embed_images.py \
    --images-dir /content/images \
    --build-dir /content/build \
    --backend dinov2 \
    --limit 200

### Sanity check the smoke vectors

Two photos of one product should score clearly higher against each other than against a random other product. If they don't, stop — something is wrong upstream.

In [ ]:
import json

import numpy as np

vectors = np.load('/content/build/embeddings_smoke.npy')
rows = [json.loads(l) for l in open('/content/build/embed_rows.jsonl')][:len(vectors)]
meta = json.load(open('/content/build/embeddings_smoke.meta.json'))
print(meta)

by_sku = {}
for i, row in enumerate(rows):
    by_sku.setdefault(row['sku'], []).append(i)
pair = next(v for v in by_sku.values() if len(v) >= 2)

same = float(vectors[pair[0]] @ vectors[pair[1]])
other = next(v[0] for k, v in by_sku.items() if v[0] not in pair)
diff = float(vectors[pair[0]] @ vectors[other])

print(f'same product : {same:.3f}')
print(f'different    : {diff:.3f}')
assert same > diff, 'Same-product photos should score higher. Stop and investigate.'
print('OK')

## 7 — The full run

Shards are written to Drive as they complete, so a disconnect costs one shard (about a
minute) rather than the whole run — just re-run this cell and it resumes.

Roughly 45–90 minutes. Keep the tab open; Colab reclaims idle sessions.

In [ ]:
!mkdir -p {DRIVE}/build
!cp /content/build/embed_rows.jsonl {DRIVE}/build/

!python scripts/rigidhitch_embed_images.py \
    --images-dir /content/images \
    --build-dir {DRIVE}/build \
    --backend dinov2 \
    --shard-size 2048

## 8 — Bring the results home

Only two small files are needed locally. The images and shards can stay behind — they
were an intermediate step and nothing downstream reads them.

Put both in your local `index_build/` folder, then run:

```
python scripts/rigidhitch_filter_and_build.py
python scripts/rigidhitch_eval_index.py
```

In [ ]:
from google.colab import files

files.download(f'{DRIVE}/build/embeddings.npy')
files.download(f'{DRIVE}/build/embeddings.meta.json')